# What kills the hand, what holds the noise, where is the facing, why does one encoder collapse
A clean, independent stage for the questions the long B runs raise. Everything here is **class-weighted B or its shadow** (no unweighted runs), D = 128,
SIGReg 0.01, one seed, the Craftax-faithful env (arrow agent, chasing mob), and every run logs the **mechanism diagnostics** (`gaf/mech.py`):
per-direction retention weights, the share of each group the floor gate protects, the gradient of each loss term on the live encoder with its
first-order effect on each part, the spectra, the decoder's and the latent's view of the facing, and the probe-size curve.

**What the long runs showed** (weighted B, D = 128): the live encoder loses the hand (shadow keeps 0.9) while position and inventory are held; the plain
run loses it too, only at ~55k steps, exactly when the retention weights fall to ~0.1 and the gate set `n_S` empties; twinkle is held for tens of
thousands of steps and then decays; the facing is absent from the shadow (0.1, flat); the live encoder's effective rank is 26 (weighted) against 120 in
the shadow, and SIGReg sees it (raw 3.2 against 1.05 healthy) but the price at weight 0.01 is only ~0.02, the size of the prediction loss.
Two mechanisms are confounded there: **retention release** (the reference map frees directions the shadow predictor cannot predict) and **open-path
pruning** (the floor-gated target path acts on every direction once the gate set empties).

| stage | runs | question |
|---|---|---|
| A | 2 x 2, 15k steps: open target path on/off (`retain_open` / `retain`) x retention as is / floor on all (`ret_floor_all`, every direction keeps weight >= 0.3) | what removes the hand, and what removes twinkle: retention release, open-path pruning, or both? |
| B | the shadow alone (arm `ae`, class-weighted), 8k steps | why is the facing not in the shadow, and why does it creep? Free of retention and prediction |
| rank | read from A | who squeezes the live encoder's spectrum, and does SIGReg lose a force balance? |

The runs go to a background process; this notebook only plots. The decision rules are written down before the results (section "Verdicts").

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results/b_mechanism"
SEED, D, WORKERS = 0, 128, 5
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
STEPS_A, STEPS_B = 15_000, 8_000
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, basis="excess", floor_steps=1500, floor_warm=True, eval_every=1000,
              eval_parts=12_000, eval_parts_steps=1000, D=D, seed=SEED, env=ENV, rec_class_power=1.0, ckpt_every=5000)
RUNS_A = {
    "open on, normal": Cfg(arm="retain_open", tag="mech-open-normal", steps=STEPS_A, log_mech=3000, **COMMON),
    "open on, floor all": Cfg(arm="retain_open", tag="mech-open-floor", steps=STEPS_A, log_mech=3000, ret_floor_all=True, ret_w_min=0.3, **COMMON),
    "open off, normal": Cfg(arm="retain", tag="mech-closed-normal", steps=STEPS_A, log_mech=3000, **COMMON),
    "open off, floor all": Cfg(arm="retain", tag="mech-closed-floor", steps=STEPS_A, log_mech=3000, ret_floor_all=True, ret_w_min=0.3, **COMMON),
}
RUN_B = {"shadow alone (ae)": Cfg(arm="ae", tag="mech-ae", steps=STEPS_B, log_mech=2000, floor_start=10 ** 9, **COMMON)}
RUNS = {**RUNS_A, **RUN_B}
PARTS = ["hand", "position", "inventory", "facing", "mob", "twinkle"]
TERMS = ["pred", "open", "ret", "sigreg", "rec"]
pd.set_option("display.precision", 3)


def hist(c):
    try:
        return pd.read_csv(f"{OUT}/{c.run_id}/metrics.csv")
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return None


def last(c, col, k=3):
    h = hist(c)
    return float(h[col].dropna().iloc[-k:].mean()) if h is not None and col in h and h[col].notna().any() else float("nan")


def panels(runs, cols, titles=None, ncols=4, ylim=None, ylabel=""):
    """One panel per column, one line per run (live encoder solid), steps on x."""
    n = len(cols)
    fig, axes = plt.subplots((n + ncols - 1) // ncols, ncols, figsize=(4.2 * ncols, 3.2 * ((n + ncols - 1) // ncols)), layout="constrained", squeeze=False)
    for (name, c), color in zip(runs.items(), A.RUN_COLORS):
        h = hist(c)
        if h is None:
            continue
        for ax, col in zip(axes.flat, cols):
            if col in h:
                ax.plot(h["step"], h[col], color=color, label=name, marker="o", ms=3)
    for ax, col in zip(axes.flat, cols):
        ax.set(title=(titles or {}).get(col, col), ylim=ylim)
    axes.flat[0].legend(fontsize=7)
    plt.show()

## Launch
Five runs in a background process (safe to re-run: finished and running runs are skipped; `A.launch` claims a run at once). Checkpoints (`ckpt_<step>.pt`)
every 5000 steps, so any later question can be asked of the weights. To stop everything: `kill -TERM -- -$(cat /workspace/results/b_mechanism/launch_*.pid)`.

In [ ]:
A.launch(list(RUNS.values()), OUT, workers=WORKERS)

In [ ]:
display(A.progress(OUT, list(RUNS.values())))

**Recovery cell.** Only needed if a run died in its final probes (a CUDA out of memory when several runs reached that phase together did this once). Such a
run has trained to the end (`ckpt_<steps>.pt` and a complete `metrics.csv`) but has no `summary.json`. This rebuilds the summary from the checkpoint, one run at a time.

In [ ]:
from pathlib import Path
for name, c in RUNS.items():
    d = Path(OUT) / c.run_id
    if d.exists() and not (d / "summary.json").exists() and (d / f"ckpt_{c.steps}.pt").exists():
        print("finalizing", name)
        A.finalize(c, OUT)

In [ ]:
A.live_probes(OUT, RUNS_A, every=30)

## Stage A: the hand and the noise
**Retention weight and protection by group.** Row 1: the live probe of the hand and of twinkle. Row 2: the retention weight the reference map puts on the
shadow directions dominated by the hand (`hard`), by position and inventory (`slow`) and by twinkle/noise, and the share of each group's live directions
that the floor gate protects (`S_frac_*`; unprotected = the open path may prune them).

In [ ]:
panels(RUNS_A, ["ev_pm_hand", "ev_pm_twinkle", "ev_sh_pm_hand", "ev_sh_pm_twinkle"], ylim=(-0.3, 1.05))
panels(RUNS_A, ["ref_w_hard", "ref_w_slow", "ref_w_noise", "ref_w_fast", "S_frac_hard", "S_frac_slow", "S_frac_noise", "S_frac_fast"], ylim=(-0.05, 1.05))

**Attribution: which loss term builds or erodes each part.** `sens_<term>_<part>` is the first-order change of the part's linear readout along the step of
that term alone, in R² points per 1000 steps (positive = builds it in the live latent, negative = erodes it). Computed every 3000 steps. The terms:
`pred` (prediction, context path), `open` (floor-gated target path, only with the open path on), `ret` (retention), `sigreg`.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 6), layout="constrained", sharey="row")
tcol = dict(zip(["pred", "open", "ret", "sigreg"], ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]))
for j, (name, c) in enumerate(RUNS_A.items()):
    h = hist(c)
    for i, part in enumerate(["hand", "twinkle"]):
        ax = axes[i, j]
        if h is not None:
            hh = h.dropna(subset=["gnorm_pred"]) if "gnorm_pred" in h else h.iloc[:0]
            for t, col in tcol.items():
                if f"sens_{t}_{part}" in hh:
                    ax.plot(hh["step"], hh[f"sens_{t}_{part}"], color=col, marker="o", ms=4, label=t)
        ax.axhline(0, color="#52514e", lw=0.8)
        ax.set(title=f"{name}: {part}", xlabel="step")
axes[0, 0].legend(fontsize=7)
axes[0, 0].set_ylabel("R² points / 1000 steps")
plt.show()

**Forces on the live encoder.** Gradient norm of each loss term, and the cosine between SIGReg and the other terms (negative = SIGReg pulls against them).

In [ ]:
panels(RUNS_A, ["gnorm_pred", "gnorm_open", "gnorm_ret", "gnorm_sigreg", "gcos_pred_sigreg", "gcos_sigreg_ret", "gcos_pred_ret", "gcos_sigreg_open"], ncols=4)

## Rank and spectra
The eigenvalues of Cov(z) of the live encoder and of the shadow at each eval (log scale) for the first A run, with the live and shadow effective rank and
the shadow's SIGReg. SIGReg sees a collapsed spectrum (raw value ~ 1.05 at full rank, 2.85 at rank 26): the question is the force balance, in the
`gnorm_*` panels above: SIGReg's gradient against prediction, open path and retention.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), layout="constrained")
name, c = next(iter(RUNS_A.items()))
try:
    d = np.load(f"{OUT}/{c.run_id}/dirs.npz")
    steps = d["step"]
    for key, ax, title in (("spec_live", axes[0], "live spectrum"), ("spec_shadow", axes[1], "shadow spectrum")):
        if key in d.files:
            for i in range(0, len(steps), max(1, len(steps) // 5)):
                ax.semilogy(np.maximum(d[key][i], 1e-8), label=f"step {steps[i]}")
            ax.set(title=f"{name}: {title}", xlabel="direction", ylim=(1e-4, None))
    axes[0].legend(fontsize=7)
except (FileNotFoundError, KeyError):
    pass
for (n_, c_), col in zip(RUNS_A.items(), A.RUN_COLORS):
    h = hist(c_)
    if h is not None:
        axes[2].plot(h["step"], h["ev_pm_rank"], color=col, label=n_)
        if "ev_sh_pm_rank" in h:
            axes[2].plot(h["step"], h["ev_sh_pm_rank"], color=col, ls="--")
axes[2].set(title="effective rank (solid live, dashed shadow)", xlabel="step")
axes[2].legend(fontsize=7)
plt.show()
panels(RUNS_A, ["sigreg", "sh_sigreg"], ncols=2)

## Stage B and the facing
Four views of the facing, none of them a plain probe: **(1) decoder orientation accuracy** (`face_acc_*`, the decoded agent tile matched to the four arrows;
chance 0.25) and the unweighted reconstruction error of the agent tiles; **(2) latent sensitivity** (`lat_sens_*`: how far the latent moves per unit of
pixel distance when only the facing / hand / position / twinkle changes, relative to two unrelated frames; about 1 = as sensitive as to a random change, far
below = the latent hardly sees it); **(3) probe-size curve** (`pc_*_facing_{lin,s,l}`: a ridge, a small and a large MLP; information that is present but
hard to read gains with size); **(4) the same for the shadow of the A runs.** Stage B is the autoencoder alone: no retention, no prediction.

In [ ]:
panels(RUN_B, ["face_acc_live", "rec_agent_live", "rec_mob_live", "rec_hand_live", "lat_sens_facing_live", "lat_sens_position_live", "lat_sens_hand_live", "lat_sens_twinkle_live"], ncols=4)
panels(RUN_B, ["pc_live_facing_lin", "pc_live_facing_s", "pc_live_facing_l", "ev_pm_facing", "pc_live_hand_lin", "pc_live_hand_l", "pc_live_mob_lin", "pc_live_mob_l"], ncols=4, ylim=(-0.3, 1.05))
panels(RUNS_A, ["face_acc_sh", "rec_agent_sh", "lat_sens_facing_sh", "pc_sh_facing_lin", "pc_sh_facing_s", "pc_sh_facing_l"], ncols=3)

## Verdicts
Rules fixed before the results. "Hand held" = live hand probe >= 0.5 as the mean of the last three evals; effects are differences between runs.

**Hand (A).** Let *floor effect* = held-ness gained by `floor all` (open path on), and *open effect* = held-ness gained by switching the open path off (normal retention).
* floor effect > 0.4: retention release is a sufficient cause. open effect > 0.4: the open path is a cause. Both: both act. Neither (< 0.2): another mechanism,
  read the attribution panel for the term with the most negative `sens_*_hand` while the hand falls.
* confirm with `ref_w_hard` (falls to < 0.1 where the hand is lost, stays >= 0.3 with the floor) and `S_frac_hard`.

**Twinkle (A).** If the runs with the open path off keep twinkle (live probe) about 0.2 higher than the matching runs with it on, the open path is what drops noise;
`S_frac_noise` falling and a negative `sens_open_twinkle` are the mechanism.

**Rank.** If `gnorm_sigreg` is below about 10 % of the other terms' norms, SIGReg is outvoted; if `gcos_sigreg_ret` is negative, retention and SIGReg conflict. (The
SIGReg 0.05 run of `05_b_d128.ipynb` is the intervention.)

**Facing (B).** Not encoded: `face_acc` about 0.25, `lat_sens_facing` far below the position/hand/twinkle values, all `pc_*_facing_*` about 0. Encoded but hard to
read: `face_acc` clearly above 0.25 while `pc_*_facing_lin` is low and `_l` is much higher. Encoded and readable: all high.

In [ ]:
def say(label, text):
    print(f"{label}: {text}")


def held(name):
    return last(RUNS_A[name], "ev_pm_hand")


names = list(RUNS_A)
if all(not np.isnan(held(n)) for n in names):
    floor_eff = held("open on, floor all") - held("open on, normal")
    open_eff = held("open off, normal") - held("open on, normal")
    print({n: round(held(n), 2) for n in names})
    say("hand, floor effect (open on)", f"{floor_eff:+.2f} -> " + ("retention release is a sufficient cause" if floor_eff > 0.4 else "no" if floor_eff < 0.2 else "partial"))
    say("hand, open-path effect (normal retention)", f"{open_eff:+.2f} -> " + ("the open path is a cause" if open_eff > 0.4 else "no" if open_eff < 0.2 else "partial"))
    tw = {n: last(RUNS_A[n], "ev_pm_twinkle") for n in names}
    print({n: round(v, 2) for n, v in tw.items()})
    say("twinkle, open path on vs off (normal retention)", f"{tw['open off, normal'] - tw['open on, normal']:+.2f}")
    say("twinkle, open path on vs off (floor all)", f"{tw['open off, floor all'] - tw['open on, floor all']:+.2f}")
else:
    print("stage A not finished")
for nm, c in {**RUN_B}.items():
    acc, sens = last(c, "face_acc_live"), last(c, "lat_sens_facing_live")
    pcl, pcs, pcL = (last(c, f"pc_live_facing_{k}") for k in ("lin", "s", "l"))
    say("facing in the shadow alone", f"decoder accuracy {acc:.2f} (chance 0.25), latent sensitivity {sens:.2f} (position {last(c, 'lat_sens_position_live'):.2f}, "
        f"hand {last(c, 'lat_sens_hand_live'):.2f}), probes lin/small/large {pcl:.2f}/{pcs:.2f}/{pcL:.2f}")

## Final readout
The probes of the end state of every run (the strong final probe).

In [ ]:
df = A.collect(OUT)
rows = {}
for name, c in RUNS.items():
    r = df[df.run_id == c.run_id]
    if len(r):
        rows[name] = {**{f"live {p}": r[f"pm_{p}"].mean() for p in PARTS}, **{f"shadow {p}": r[f"sh_pm_{p}"].mean() for p in PARTS if f"sh_pm_{p}" in r and r[f"sh_pm_{p}"].notna().any()}}
display(pd.DataFrame(rows).T.round(3))

## Decoder-strength readout
The probes under-read compressed latents (the held hand: 0.46 on the eval probe, 0.82 on the final one, same step; the facing: ~0.1 on every probe while the
shadow's decoder renders its orientation 95 % of the time). This reads what a latent holds at **decoder strength**: freeze the live encoder and the shadow of the
last checkpoint, train a fresh class-weighted decoder on each (5000 steps), and report per tile class the share of the error that a decoder knowing nothing
about that content would make, removed by the decoded frame (0 = nothing decoded, 1 = perfect), plus the facing orientation accuracy of the decoded agent tile
(chance 0.25). Compare the shadow rows with the live rows: content present in the shadow and not in the live row was released by retention; content absent from
both was never encoded. About 1-2 minutes per decoder on a GPU, 8 decoders. Needs the checkpoints at the last step (`ckpt_every`).

In [ ]:
import torch, pandas as pd
from pathlib import Path
from types import SimpleNamespace
from gaf.models import Encoder, Decoder
from gaf.train import make_env, device, class_weights, rec_error
from gaf import mech

def decoder_strength(c, out_dir, steps=5000, n_eval=2048):
    """Freeze the live encoder and the shadow of the checkpoint at c.steps, train a fresh decoder on each latent, and report per tile class
    the share of the error a decoder that knows nothing about that content would make, removed by the decoded frame (1 - mse / mse of the
    average image at that tile position; 0 = nothing decoded, 1 = perfect), plus the facing orientation accuracy of the decoded agent
    tile (chance 0.25). A readout at decoder strength: it finds content a small probe cannot."""
    d, dev = Path(out_dir) / c.run_id, device()
    env = make_env(c.env, dev)
    sd = torch.load(d / f"ckpt_{c.steps}.pt", map_location=dev)
    w, _ = class_weights(env, 1.0)
    t, W = c.env.tile, c.env.W
    clip = env.sample(n_eval)
    tiles = clip.x[:, -1].view(n_eval, 3, W, t, W, t).permute(0, 2, 4, 1, 3, 5).reshape(n_eval, W * W, -1)
    cls = clip.f["tile_class"][:, -1]
    dev2 = ((tiles - tiles.mean(0, keepdim=True)) ** 2).mean(-1)  # per tile: error of the average image at its position
    base = {}
    for name, ids in mech.REC_GROUPS.items():
        m_ = torch.isin(cls, torch.tensor(list(ids), device=dev))
        base[name] = dev2[m_].mean().item() if m_.any() else float("nan")
    res = {}
    for tag, key in (("live", "enc"), ("shadow", "shadow")):
        if key not in sd:
            continue
        enc = Encoder(c.D, c.env.res).to(dev); enc.load_state_dict(sd[key]); enc.eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        dec = Decoder(c.D, c.env.res).to(dev)
        opt = torch.optim.AdamW(dec.parameters(), 1e-3, weight_decay=0.01)
        for _ in range(steps):
            cl = env.sample(256)
            with torch.no_grad():
                z = enc(cl.x[:, -1])
            loss = rec_error(dec(z), cl.x[:, -1], cl.f["tile_class"][:, -1], w, t)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        m = mech.decoder_view(SimpleNamespace(env=env, cfg=c), enc, dec, tag)
        res[tag] = {**{n: 1 - m[f"rec_{n}_{tag}"] / base[n] for n in base if base[n] == base[n] and base[n] > 0}, "facing acc": m.get(f"face_acc_{tag}")}
    return res

rows = {}
for name, c in RUNS_A.items():
    print("decoding", name, flush=True)
    for tag, vals in decoder_strength(c, OUT).items():
        rows[(name, tag)] = vals
DS = pd.DataFrame(rows).T
display(DS.round(2))